In [0]:
%pip install sqlalchemy psycopg2-binary pandas
dbutils.library.restartPython()

In [0]:
POSTGRES_HOST = "ep-summer-tooth-a8hwz3t3-pooler.eastus2.azure.neon.tech"
POSTGRES_PORT = "5432"
POSTGRES_DB = "Demo-Database"
POSTGRES_USER = "neondb_owner"
POSTGRES_PASSWORD = "npg_QU4WcsN2EXSi"

In [0]:
from urllib.parse import quote_plus

POSTGRES_URL = (
    f"postgresql+psycopg2://{POSTGRES_USER}:{quote_plus(POSTGRES_PASSWORD)}"
    f"@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}?sslmode=require"
)

print(POSTGRES_URL.replace(quote_plus(POSTGRES_PASSWORD), "****"))

In [0]:
from sqlalchemy import create_engine, text

engine = create_engine(POSTGRES_URL)

with engine.connect() as conn:
    result = conn.execute(text("SELECT version(), current_database(), current_user;"))
    for row in result:
        print(row)

In [0]:
create_sql = """
CREATE TABLE IF NOT EXISTS lab_test_neon (
    id SERIAL PRIMARY KEY,
    source_name VARCHAR(100),
    status VARCHAR(30),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
"""
with engine.connect() as conn:
    conn.execute(text(create_sql))
    conn.commit()

print("Tabla creada")

In [0]:
insert_sql = """
INSERT INTO lab_test_neon (source_name, status)
VALUES (:source_name, :status)
"""

with engine.connect() as conn:
    conn.execute(
        text(insert_sql),
        {"source_name": "databricks", "status": "ok"}
    )
    conn.commit()

print("Registro insertado")

In [0]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM lab_test_neon ORDER BY id DESC LIMIT 10"))
    for row in result:
        print(row)

In [0]:
# Definimos la consulta SQL para borrar la tabla de forma segura
drop_sql = """
DROP TABLE IF EXISTS lab_test_neon;
"""

# Usamos el motor para abrir la conexión
with engine.connect() as conn:
    # Ejecutamos el comando SQL
    conn.execute(text(drop_sql))
    
    # ¡Importante! Hacemos commit para que el borrado sea permanente en PostgreSQL
    conn.commit()

print("Tabla lab_test_neon eliminada correctamente")